## Импорт библиотек

In [11]:
import pandas as pd
import sqlite3

## Создание подключения к базе данных

In [12]:
conn = sqlite3.connect('data/checking-logs.sqlite')

## Расчет средней разницы до и после первого просмотра новостной ленты для тестовой группы

In [13]:
test_results = pd.io.sql.read_sql("""
    WITH user_data AS (
        SELECT uid, first_commit_ts, deadlines,
               CASE WHEN first_commit_ts < first_view_ts THEN 'before' ELSE 'after' END AS time
        FROM test
        JOIN deadlines ON labname = labs
        WHERE labname != 'project1'
    )
    SELECT time,
           AVG((strftime('%s', first_commit_ts) - deadlines) / 3600) AS avg_diff
    FROM user_data
    WHERE uid IN (SELECT uid FROM user_data GROUP BY uid HAVING COUNT(DISTINCT time) = 2)
    GROUP BY time
""", conn)
test_results

,time,avg_diff
0,after,-104.6000
1,before,-60.5625


## Расчет средней разницы до и после первого просмотра новостной ленты для контрольной группы

In [14]:
control_results = pd.io.sql.read_sql("""
    WITH user_data AS (
        SELECT uid, first_commit_ts, deadlines,
               CASE WHEN first_commit_ts < first_view_ts THEN 'before' ELSE 'after' END AS time
        FROM control
        JOIN deadlines ON labname = labs
        WHERE labname != 'project1'
    )
    SELECT time,
           AVG((strftime('%s', first_commit_ts) - deadlines) / 3600) AS avg_diff
    FROM user_data
    WHERE uid IN (SELECT uid FROM user_data GROUP BY uid HAVING COUNT(DISTINCT time) = 2)
    GROUP BY time
""", conn)
control_results

,time,avg_diff
0,after,-117.636364
1,before,-99.464286


## Закрытие соединения

In [15]:
conn.close()

**Вопрос:** Оказалась ли гипотеза верна, и страница влияет на поведение студентов?

**Ответ:** В тестовой группе мы получили: средняя разница до просмотра новостой ленты -60.5625, после - -104.6 (-104.6 + 60.5625 = -44.0375);
В контрольной группе мы получили: средняя разница до просмотра новостой ленты -99.464286, после - -117.636364 (-117.636364 + 99.464286 = 18.172078);
Соответственно, для тестовой группы изменение более чем в два раза выше, чем в случае контрольной. Значит, новостная лента влияет на поведение студентов, и гипотеза **подтверждается**.